**Random forests extend bagging by introducing additional randomness through feature subsampling at each tree node, further decorrelating base decision trees to reduce variance while maintaining low bias, resulting in robust ensembles for classification and regression tasks.**

## Introduction to Random Forests
Random forests represent an evolution of bagging ensembles, specifically tailored to decision tree bases. By combining bootstrap aggregation with random feature selection, they mitigate the high variance and correlation inherent in individual trees, yielding models that generalise better than single CARTs or standard bagging. This dual randomness—over instances and features—promotes diversity, enabling the ensemble to capture complex patterns without overfitting.

Unlike voting classifiers (diverse algorithms, same data) or pure bagging (same algorithm, bootstrapped data, all features), random forests constrain feature access per split, reducing tree similarity. This makes them versatile for non-linear dependencies, requiring no preprocessing like scaling, and providing built-in feature importance metrics for interpretability.

---

## Theoretical Foundations
Random forests leverage the bias-variance decomposition to optimise generalisation. A single decision tree often exhibits low bias but high variance, overfitting noise. Bagging reduces variance by averaging bootstrapped trees, but correlated trees (from using all features) limit gains. Random forests decorrelate by subsampling $d$ features without replacement at each node, where $d < p$ (total features), typically $d = \sqrt{p}$ for classification or $d = p/3$ for regression.

The variance reduction for an ensemble of $B$ trees is:
$$
\text{Var}[\bar{f}] \approx \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2,
$$
where $\rho$ is tree correlation, $\sigma^2$ individual variance. Feature subsampling lowers $\rho$, amplifying the $(1 - \rho)/B$ term.

Bootstrap sampling ensures each tree sees ~63% unique instances, with OOB for validation. At each node, the split maximises impurity reduction (Gini/entropy for classification, MSE for regression) over subsampled features, approximating oblique boundaries through aggregation.

Theoretically, as $B \to \infty$ and $d$ tuned, random forests converge to Bayes error bounds under weak assumptions, outperforming single models on noisy, high-dimensional data.

---

## Training Process
Training involves generating $B$ decision trees:
1. Draw bootstrap sample of size $N$ with replacement.
2. For each tree:
   - At each node, subsample $d$ features without replacement.
   - Select split maximising information gain over subsampled features.
   - Grow unconstrained (or with minimal constraints) to maximise diversity.
3. Repeat for all $B$ trees.

This process decorrelates trees: bootstraps vary instances, subsampling varies features. For $p$ features, probability a feature is excluded per split is $(1 - d/p)^p \approx e^{-d}$, ensuring exploration.

Diagram recreation code:

```python
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, ArrowStyle, FancyArrowPatch

# Figure
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

# Training set
ax.add_patch(Rectangle((4, 0.5), 2, 1, fill=True, color='lightgrey'))
ax.text(5, 1, 'Training Set', ha='center', va='center')

# Bootstrap samples
ax.text(5, 2, 'Bootstrap Samples (size = N)', ha='center', va='center', color='blue')

# Trees
x_pos = [2, 4, 6, 8]
for i in range(4):
    if i < 1:
        ax.add_patch(Rectangle((x_pos[i] - 0.5, 4.5), 1, 1, fill=True, color='lightgreen'))
        ax.text(x_pos[i], 5, 'Tree 1', ha='center', va='center')
    elif i == 1:
        ax.text(x_pos[i], 5, '...', ha='center', va='center')
    elif i == 2:
        ax.add_patch(Rectangle((x_pos[i] - 0.5, 4.5), 1, 1, fill=True, color='lightgreen'))
        ax.text(x_pos[i], 5, 'Tree K', ha='center', va='center')
    else:
        continue

# Feature subsampling
ax.text(3, 6, 'Sample d features at each split (without replacement)', ha='center', va='center', color='orange')

# Arrows
for x in [2, 6]:
    arrow = FancyArrowPatch((x, 4.5), (x, 3), arrowstyle=ArrowStyle('->', head_width=3), color='blue')
    ax.add_patch(arrow)

ax.set_title('Random Forest Training Process')
plt.show()
```

This illustrates bootstrapping and feature randomness.

---

## Prediction Process
Predictions aggregate base outputs. For classification:
$$
\bar{f}(\mathbf{x}) = \arg\max_k \frac{1}{B} \sum_{b=1}^B \mathbb{I}(\hat{f}_b(\mathbf{x}) = k).
$$
For regression:
$$
\bar{f}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^B \hat{f}_b(\mathbf{x}).
$$

Soft voting (probabilities) enhances calibration. Aggregation smooths discontinuities, improving robustness.

---

## Feature Importance
Random forests quantify feature contributions via mean impurity decrease. For feature $j$, importance $I_j$ is the weighted sum of impurity reductions across nodes splitting on $j$:
$$
I_j = \sum_{t \in T} w_t \cdot \Delta i_t(j),
$$
where $T$ is all nodes, $w_t$ node weight (proportion of samples), $\Delta i_t(j)$ reduction from splitting on $j$. Normalised: $\sum_j I_j = 1$.

This measures predictive power, aiding selection and interpretation. Permutation importance alternatives assess drop in OOB score post-shuffling.

Using Diabetes dataset for regression:

```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

# Load data
data = load_diabetes()
X, y = data.data, data.target
feature_names = data.feature_names

# Train RF
rf = RandomForestRegressor(n_estimators=400, random_state=1)
rf.fit(X, y)

# Importances
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values()

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, color='lightgreen', ax=ax)
ax.set_title('Feature Importances')
ax.set_xlabel('Mean Impurity Decrease')
plt.show()
```

This barplot ranks features by contribution.

---

## Implementation in Scikit-Learn
`RandomForestClassifier` and `RandomForestRegressor` encapsulate the process, with `n_estimators=B`, `max_features=d` (default sqrt(p) for classification, p/3 for regression).

For regression on Diabetes:

```python
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Load
data = load_diabetes()
X, y = data.data, data.target

# Split
SEED = 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED
)

# RF
rf = RandomForestRegressor(
    n_estimators=400, min_samples_leaf=0.12, random_state=SEED
)

# Fit, predict
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# RMSE
rmse = mean_squared_error(y_test, y_pred) ** 0.5
print(f"Test RMSE: {rmse:.2f}")
```

This demonstrates lower error than single trees.

---

## Practical Considerations and Extensions
Random forests resist overfitting with large $B$, but tune $d$ and leaf constraints via OOB or CV. They handle mixed data types, missing values (via surrogates), but scale quadratically with $N$.

Extensions include extremely randomised trees (random thresholds). Theoretically, consistency holds under bootstrap and subsampling conditions, converging to optimal predictors. In practice, parallelise with `n_jobs=-1` for efficiency.